# FINA4030A — Lab 7
## Auditing an auditor

**Class 7.** **Not assessed.** Submit it anyway — it is the last thing you do
before the Class 8 exercise, and it is the closest rehearsal for it you will get.

> **Before you type anything: File → Save a copy in Drive.**

A macro desk asked for a hawkish/dovish index built from eleven years of FOMC
statements and minutes, and a test of whether it predicts the 2s10s curve.

**This session was designed around a trap.** FOMC minutes are published *three
weeks after* the meeting they describe; statements go out the same afternoon.
A corpus keyed on the date printed on the document therefore hands a backtest
twenty-one days of future information, eight times a year, and nothing about the
file looks wrong. Every document you have been given carries both dates.

The system did not fall for it. It found the lag, quantified what using the wrong
date would have bought — an eight-point rise in the hit rate — avoided it, and
warned the desk that anyone else's strong result on this data is probably
contaminated.

Then it reported a positive result and spent the rest of its report arguing that
you should not believe it.

**So your job is not to find an error. It is to decide whether the scepticism is
correct — which is a different question from whether it is impressive.**

**Five passes.**

1. **Reproduce the trap.** Its central claim, checked.
2. **Verify the scepticism.** Eleven specific numbers. Do they hold?
3. **Find what it did not say.** There is one thing.
4. **Decide whether that matters.** This is the hard step, and most of you will
   get it wrong in a particular direction.
5. **Price it.** The last section is the one to carry into Class 8.


In [ ]:
# Setup. Run this first. The corpus is about 6 MB.
REQUIRED_CLIENT = "1.1"
REPO = "https://raw.githubusercontent.com/fy-ericlam/fina4030a/main"

import importlib, sys, urllib.request, warnings
warnings.filterwarnings("ignore")

urllib.request.urlretrieve(f"{REPO}/fina4030a.py", "fina4030a.py")
for f in ("lab07_fomc.json", "lab07_tone.py", "lab07_tone_report.txt",
          "lab07_request.txt"):
    urllib.request.urlretrieve(f"{REPO}/labs/{f}", f)

sys.modules.pop("fina4030a", None)
import fina4030a
importlib.reload(fina4030a)
assert fina4030a.__version__ >= REQUIRED_CLIENT, (
    f"client v{fina4030a.__version__}, this lab needs v{REQUIRED_CLIENT} -- "
    "re-run this cell; if it persists, Runtime > Restart session")
print(f"client v{fina4030a.__version__} loaded")

NAME       = ""
STUDENT_ID = ""

fina4030a.configure(provider="cuhk_portal")
fina4030a.verify()


---
## What was asked, and what came back


In [ ]:
print(open("lab07_request.txt", encoding="utf-8").read())


In [ ]:
# Its report. Read it properly -- this is the document you are auditing.
report = open("lab07_tone_report.txt", encoding="utf-8").read()
print(report)


---

## Calibration log — the top half now, before Pass 1

Fill the **before** fields in this cell now, while you still do not know the
answer. **Nothing here is marked** — this lab is not assessed. But Class 8
awards **25 of its 100 marks** for calibration, and this is the last time you
record a prior before doing it under supervision, for 30% of the course.

Come back for the **after** fields at the end: edit the cell and run it again.


In [ ]:
CALIBRATION = {
    # --- Before Pass 1. You have read the request and the report; that is enough. ---
    # The report is candid about its own choices. How many material things do you
    # expect to find that it has NOT already disclosed?
    "expected_undisclosed_findings": None,   # 0-5
    # How far would you trust its conclusion right now?
    "trust_before":                  None,   # 1 = not at all, 5 = would sign it
    # --- After. ---
    "actual_undisclosed_findings":   None,   # 0-5
    "trust_after":                   None,   # 1-5, same scale
    "surprise":                      "",     # one sentence: what you did not expect
}

_todo = [k for k, v in CALIBRATION.items()
         if v is None or (isinstance(v, str) and not v.strip())]
print("Calibration log complete."
      if not _todo else "Still to fill in: " + ", ".join(_todo))


---
# Pass 1 — reproduce the trap

Its central factual claim is that timing minutes on the meeting date rather than
the release date inflates the ten-day hit rate from **54.4% to 62.5%**.

Check it. The specimen is importable and its functions are the ones it used.


In [ ]:
import numpy as np, pandas as pd
import lab07_tone as T

raw, docs, y = T.load("lab07_fomc.json")
idx = T.build_index(docs).sort_values("release_date").reset_index(drop=True)
print(f"{len(idx)} documents, {(idx.kind=='statement').sum()} statements, "
      f"{(idx.kind=='minutes').sum()} minutes")

minutes = idx[idx.kind == "minutes"]
for col in ("meeting_date", "release_date"):
    al = T.align(minutes, y, col)
    pos = al["t0_pos"].values
    row = f"  minutes timed on {col:<13}"
    for h in (10, 21):
        f2 = T.forward_change(y, pos, h)
        hb = T.hit_block(al["tone_z"].values, f2, np.mean(f2[np.isfinite(f2)] < 0))
        row += f"   h={h}: {hb['hit']:.3f}"
    print(row)


That is the whole session's original subject, found and closed by the system
before anybody asked it to.

**One thing it did not mention.** It joined on the recorded `release_date` rather
than computing one — which is correct, and which means it never had to know how
long the lag actually is. Check the lag yourself.


In [ ]:
from collections import Counter
lags = Counter(idx[idx.kind == "minutes"]["release_lag_days"])
print("minutes release lag, days after the meeting:")
for k in sorted(lags):
    print(f"   {k:>3} days   {'#' * lags[k]}  ({lags[k]})")
off = sum(v for k, v in lags.items() if k != 21)
print(f"\n{off} of {sum(lags.values())} meetings ({100*off/sum(lags.values()):.0f}%) "
      f"are NOT exactly 21 days.")


In [ ]:
PASS1 = {
    "leak_reproduces":      None,   # True / False
    "hit_meeting_date":     None,   # the number you got
    "hit_release_date":     None,
    "why_21_days_is_not_a_fix": "",  # someone "fixes" look-ahead by adding 21
                                     # days to the meeting date. What is wrong
                                     # with that, and how often?
}
_m = [k for k, v in PASS1.items() if v is None or (isinstance(v, str) and not v.strip())]
print("Pass 1 complete." if not _m else "Still to fill: " + ", ".join(_m))


---
# Pass 2 — is the scepticism correct?

The report makes eleven checkable numerical claims. Impressive is not the same as
correct, and the only way to tell them apart is to recompute.

Fill in `CLAIMS` with what you get. One of them does not match.


In [ ]:
al = T.align(idx, y, "release_date")
pos, sig = al["t0_pos"].values, al["tone_z"].values
f21 = T.forward_change(y, pos, 21)
hb21 = T.hit_block(sig, f21, np.mean(f21[np.isfinite(f21)] < 0))

print(f"  headline hit rate h=21      {hb21['hit']:.3f}   (report: 0.598)")
print(f"  base rate                   {hb21['base']:.3f}   (report: 0.500)")
print(f"  naive binomial p            {hb21['p']:.3f}   (report: 0.010)")
print(f"  circular-shift p            {T.circ_p(sig, f21, n_iter=20000, seed=0):.3f}"
      f"   (report: 0.113)  <- its own settings: 20,000 shifts, seed 0")
s = al['tone_z'].dropna().values
rho = np.corrcoef(s[:-1], s[1:])[0, 1]
print(f"  tone lag-1 autocorrelation  {rho:.2f}   (report: 0.77)")
print(f"  effective sample size       {len(s)*(1-rho)/(1+rho):.0f} of {len(al)}"
      f"   (report: ~24 of 187)")

# The same-day regression: does the text move the curve on the day it lands?
# Its standard error is Newey-West, so recomputing it means using the same
# estimator -- what you are checking here is the arithmetic, not the choice.
rx = T.reaction_change(y, pos)
mm = (al.kind == "minutes").values
print(f"  same-day reaction t         "
      f"{T.newey_west_t(al['tone_z'].values[mm], rx[mm], 1)[1]:+.2f}"
      f"   (report: -0.68, minutes)")

fin = f21[np.isfinite(f21)]
pnl = -np.sign(sig[np.isfinite(f21)]) * fin
print(f"  21-day 2s10s sd             {fin.std(ddof=1):.1f} bp   (report: 15.7)")
print(f"  mean signed move            {pnl.mean():+.1f} bp   (report: +1.2)")
print(f"  t-stat                      {pnl.mean()/(pnl.std(ddof=1)/np.sqrt(len(pnl))):.2f}"
      f"   (report: 1.05)")


Now the claim about the early sample. The report says *"Through 2018, 8 of 10
signal/horizon cells are negative."* Count them.


In [ ]:
early = (al["release_date"].dt.year <= 2018).values
neg = 0
for sn in T.SIGNALS:
    for h in T.HORIZONS:
        f2 = T.forward_change(y, pos, h)
        hb = T.hit_block(al[sn].values[early], f2[early],
                         np.mean(f2[np.isfinite(f2)] < 0))
        neg += hb["edge"] < 0
        print(f"   {sn:<12} h={h:<3} edge {hb['edge']:+.3f}")
print(f"\nnegative cells: {neg} of 10   (report says 8)")


In [ ]:
PASS2 = {
    "claims_checked":        None,  # integer
    "claims_that_held":      None,  # integer
    "the_one_that_did_not":  "",    # which, and what the right number is
    "does_it_favour_or_undermine_the_report": "",   # think about the direction
}
_m = [k for k, v in PASS2.items() if v is None or (isinstance(v, str) and not v.strip())]
print("Pass 2 complete." if not _m else "Still to fill: " + ", ".join(_m))


---
# Pass 3 — find what it did not say

Everything above held, and the one discrepancy made its own case *weaker*. So the
report is honest. That is exactly when to look hardest.

The report is candid about its choices. The **code** contains one that the report
never mentions, and it is a form of look-ahead.

Read `build_index` and think about *when* each quantity could have been known.


In [ ]:
import inspect
print(inspect.getsource(T.build_index))


Now find what it puts at risk. Section 4 of the report offers **two** things in
the signal's favour — the jackknife in §4(d) and the walk-forward in §4(f) — and
§4(d) calls itself *"the one point in the signal's favour"* while §4(f) then adds
another. Notice that. A report that contradicts itself about how much evidence it
has is not lying, but it is a report written in one pass and not re-read.


In [ ]:
PASS3 = {
    "the_undisclosed_choice": "",   # name it, and quote the line
    "why_it_is_look_ahead":   "",   # what does an observation know that it should not?
    "which_results_it_could_contaminate": "",  # both of section 4's favourable
                                               # results are downstream of it.
                                               # For each: how would it bite?
}
_m = [k for k, v in PASS3.items() if not str(v).strip()]
print("Pass 3 complete." if not _m else "Still to fill: " + ", ".join(_m))


---
# Pass 4 — does it matter?

You have found an undisclosed leak sitting upstream of both of the report's
positive results. The temptation now is enormous and you should notice yourself
feeling it: you want the walk-forward to collapse.

**Test it instead.** Rebuild the index with *causal* standardisation — at each
document, use only the mean and variance of documents released up to that point —
and re-run both.


In [ ]:
def causal_z(s):
    """Point-in-time z-score: expanding mean and sd, no future."""
    return (s - s.expanding().mean()) / s.expanding().std(ddof=0).replace(0, np.nan)

c = idx.copy()
c["tone_zc"] = c.groupby("kind")["tone"].transform(causal_z)
c["tone_zc_chg"] = c.groupby("kind")["tone_zc"].diff()
alc = T.align(c, y, "release_date")

def walk_forward(frame, cols, causal_base=False):
    """Pick signal and horizon on the first half; apply blind to the second."""
    p = frame["t0_pos"].values          # this frame's own event positions
    n = len(frame); tr = np.arange(n) < n // 2; te = ~tr
    best, be = None, -1
    for sn in cols:
        for h in T.HORIZONS:
            f2 = T.forward_change(y, p, h)
            bd = (np.mean(f2[tr][np.isfinite(f2[tr])] < 0) if causal_base
                  else np.mean(f2[np.isfinite(f2)] < 0))
            hb = T.hit_block(frame[sn].values[tr], f2[tr], bd)
            if np.isfinite(hb["edge"]) and hb["edge"] > be:
                best, be = (sn, h), hb["edge"]
    sn, h = best
    f2 = T.forward_change(y, p, h)
    bd = (np.mean(f2[tr][np.isfinite(f2[tr])] < 0) if causal_base
          else np.mean(f2[np.isfinite(f2)] < 0))
    return sn, h, T.hit_block(frame[sn].values[te], f2[te], bd)

for label, fr, cols, cb in (
        ("as delivered (full-sample z)", al,  ("tone_z", "tone_z_chg"), False),
        ("causal z",                     alc, ("tone_zc", "tone_zc_chg"), False),
        ("causal z + causal base",       alc, ("tone_zc", "tone_zc_chg"), True)):
    sn, h, hb = walk_forward(fr, cols, cb)
    print(f"  {label:<30} picked {sn} h={h:<3}  n={hb['n']:>3}  "
          f"hit {hb['hit']:.3f}  edge {hb['edge']:+.3f}  p={hb['p']:.3f}")


And the other one: the jackknife, dropping one calendar year at a time. The
report gets a range of +0.069 to +0.121 and reads it as "no single year drives
it". Re-run it on the causal index.


In [ ]:
def jackknife(frame, col, h=21):
    p = frame["t0_pos"].values
    f2 = T.forward_change(y, p, h)
    base = np.mean(f2[np.isfinite(f2)] < 0)
    yrs = frame["release_date"].dt.year
    out = {}
    for yr in sorted(yrs.unique()):
        k = (yrs != yr).values
        out[yr] = T.hit_block(frame[col].values[k], f2[k], base)["edge"]
    return out

for label, fr, col in (("as delivered", al, "tone_z"), ("causal z", alc, "tone_zc")):
    j = jackknife(fr, col)
    print(f"  {label:<14} edge range {min(j.values()):+.3f} to {max(j.values()):+.3f}"
          f"   (report: +0.069 to +0.121)")


Now work out **why** you got that answer, because the mechanism is the lesson and
the number is not.

The hit-rate test uses only `sign(signal)`. Ask what a z-score does to a sign.


In [ ]:
a, b = idx["tone_z"].values, c["tone_zc"].values
m = np.isfinite(a) & np.isfinite(b)
print(f"signs that differ between full-sample and causal z: "
      f"{(np.sign(a[m]) != np.sign(b[m])).sum()} of {m.sum()} "
      f"({100*(np.sign(a[m]) != np.sign(b[m])).sum()/m.sum():.1f}%)")
print(f"correlation between the two: {np.corrcoef(a[m], b[m])[0,1]:.3f}")


In [ ]:
VERDICT = {
    "verdict":  "",     # wrong | undisclosed but immaterial | unsupported
    "evidence": "",     # the numbers, both variants
    "mechanism": "",    # WHY it does not bite -- one sentence, about sign()
    "what_you_would_write_to_the_author": "",   # you still have to raise it.
                                                # How, given it changes nothing?
}
_m = [k for k, v in VERDICT.items() if not str(v).strip()]
print("Complete." if not _m else "Still to fill: " + ", ".join(_m))


**If you wrote "wrong", go back.** A defect that changes no reported number is
*undisclosed*, not *wrong*, and an auditor who cannot hold that distinction will
either cry wolf or wave things through. Both are firing offences and the second
one is worse.

It still has to be raised. Undisclosed leakage that happens to be inert today
becomes live the moment someone changes the test — swap the hit rate for a
regression coefficient and the full-sample mean is suddenly load-bearing.


---
# Pass 5 — price it

Everything so far has been statistics. This part is the job.

The signal survives every test the report throws at it, weakly. So: would you
trade it?


In [ ]:
sd = fin.std(ddof=1)
edge = pnl.mean()
print(f"  mean signed move per release   {edge:+.2f} bp")
print(f"  standard deviation of the move {sd:.1f} bp")
print(f"  t-statistic                    {edge/(pnl.std(ddof=1)/np.sqrt(len(pnl))):.2f}")
print(f"  releases per year              {len(pnl)/((idx.release_date.max()-idx.release_date.min()).days/365.25):.0f}")

COST_BP = None   # <-- your assumption: round-trip cost of a 2s10s spread trade,
                 #     in basis points. Look at the number above it first.
if COST_BP is not None:
    print(f"\n  net edge after {COST_BP} bp of cost: {edge - COST_BP:+.2f} bp per release")
    print("  " + ("still positive" if edge > COST_BP else
                  "negative -- the strategy pays the street and not you"))


In [ ]:
ECONOMICS = {
    "cost_assumption_bp":   None,  # what you used, and be honest about the source
    "net_edge_bp":          None,
    "would_you_trade_it":   "",    # and at what size
    "is_the_signal_real":   "",    # separate question from the one above
}
_m = [k for k, v in ECONOMICS.items() if v is None or (isinstance(v, str) and not v.strip())]
print("Complete." if not _m else "Still to fill: " + ", ".join(_m))


---
## Now ask the model

You have audited a report written by a system. Give the same report to a system
and ask what it would raise.


In [ ]:
PROMPT = """Below is a research report on a text-derived trading signal. The
author argues against their own headline result at length.

Your task is not to summarise it. It is to find what the report does NOT say.
Identify any methodological choice that is made in the analysis but not disclosed
or discussed in the report, and for each one state whether it would change the
reported conclusions, and how you would test that.

REPORT:
""" + report[:9000]

review = fina4030a.complete(PROMPT, temperature=0.0, max_tokens=1000)
print(review)


In [ ]:
MODEL_REVIEW = {
    "did_it_find_the_standardisation": None,  # True / False
    "did_it_say_whether_that_matters": "",    # or did it just flag it?
    "anything_it_found_you_missed":    "",    # or "nothing"
    "anything_it_asserted_wrongly":    "",    # check before answering
}
_m = [k for k, v in MODEL_REVIEW.items()
      if v is None or (isinstance(v, str) and not v.strip())]
print("Complete." if not _m else "Still to fill: " + ", ".join(_m))


---
## Findings


In [ ]:
FINDINGS = {
    "was_the_scepticism_correct": "",  # the report's own case against itself
    "what_it_missed":             "",
    "did_that_change_anything":   "",
    "statistically":              "",  # is there a signal?
    "commercially":               "",  # is there a trade?
    "confidence":                 None,  # 1-5
}
_m = [k for k, v in FINDINGS.items()
      if v is None or (isinstance(v, str) and not v.strip())]
print("Complete." if not _m else "Still to fill: " + ", ".join(_m))

# Calibration log, checked here too — the "after" fields are easy to forget.
if "CALIBRATION" not in globals():
    print("Calibration log: run the calibration cell above before you submit.")
else:
    _cal = [k for k, v in CALIBRATION.items()
            if v is None or (isinstance(v, str) and not v.strip())]
    print("Calibration log complete."
          if not _cal else "Calibration log still to fill in: " + ", ".join(_cal))


In [ ]:
print(fina4030a.appendix(
    student=f"{NAME} ({STUDENT_ID})",
    verification=FINDINGS.get("what_it_missed", ""),
    residual_risk=FINDINGS.get("did_that_change_anything", ""),
    reproducibility=(
        f"Audit of lab07_tone.py against lab07_fomc.json "
        f"({len(idx)} FOMC documents, {idx.release_date.min():%Y-%m-%d} to "
        f"{idx.release_date.max():%Y-%m-%d}; {len(y)} yield sessions). "
        f"Walk-forward re-run under causal standardisation."),
))
fina4030a.save_transcript("lab07_transcript.json")


---

## What to submit

This notebook with outputs intact, plus `lab07_transcript.json`. Not assessed —
but it is the last work you do before Class 8, and Class 8 is 30%.

## Two things to carry into next week

**The first is about scepticism.** The report you audited was better than most
human research you will read. It found a trap nobody warned it about, quantified
it, and then argued against its own result harder than a referee would. And it
still left one thing undisclosed.

The lesson is not that it failed. It is that **being impressed and being
satisfied are different states**, and only one of them is a review. You verified
eleven numbers before you were entitled to an opinion about the twelfth.

**The second is about what a result is worth.** You ended with a signal that is
statistically defensible and commercially worthless: real by every test the data
can support, and smaller than the cost of acting on it. Most quantitative
findings are this shape. Nothing else in your degree tells you so, because
coursework results are built to be significant.

> A number can be right, reproducible, honestly obtained, and still not a reason
> to do anything.

---

### Next

**Class 8** is the Delegation and Verification Exercise: in class, individual,
supervised, 165 minutes, 30% of the course.

Everything you did today — reproduce the claim, verify the numbers, find the
undisclosed choice, decide whether it matters, price the result — is what that
exercise asks of you, on material you have not seen, without the internet.

Today's lab was open on purpose. The defect at its centre is one a capable system
finds in a single request, which is precisely why it cannot be examined. What
Class 8 examines is the part no system did for you: deciding what the finding was
worth.


---

# Extension — only if you have finished

Everything above is one kind of look-ahead: **one document, two dates**, and a
join on the wrong one. There is a second kind that the Week 7 framework names and
this lab has not touched — **one date, two values.**

The Hong Kong Monetary Authority publishes, each business day, a *forecast* of
the next day's Aggregate Balance. The settled figure follows. Both describe the
same date. Only one of them existed the day before.

That is the same defect wearing different clothes, and it is harder, because
nothing in the file is called `release_date`. You have to notice that a column
named `aggregate_balance` is a number from the future.


In [ ]:
import json
urllib.request.urlretrieve(f"{REPO}/labs/lab07_hkma.json", "lab07_hkma.json")
H = json.load(open("lab07_hkma.json", encoding="utf-8"))
print(H["note"], "\n")

hk = pd.DataFrame(H["balances"])
hk["date"] = pd.to_datetime(hk["date"])
hk = hk.dropna(subset=["aggregate_balance", "forecast_for_today_made_yesterday"])
print(f"{len(hk)} business days, {hk.date.min():%Y-%m-%d} to {hk.date.max():%Y-%m-%d}\n")
print(hk[["date", "forecast_made_on", "forecast_for_today_made_yesterday",
          "aggregate_balance"]].head(4).to_string(index=False))


First: how different are the two, and in which direction?

**Take the median over the days that differ, not over every day.** Most days the
forecast is exactly right, and a median over all of them tells you about the
quiet days rather than about the risk.


In [ ]:
rev = hk["aggregate_balance"] - hk["forecast_for_today_made_yesterday"]
moved = rev[rev.abs() > 5]                    # 5 HK$m: below that it is rounding

print(f"  days the settled figure differs from the forecast  "
      f"{len(moved)} of {len(rev)}  ({100*len(moved)/len(rev):.0f}%)")
print(f"  median revision ON THOSE DAYS                     "
      f"{moved.abs().median():>12,.0f} HK$m")
print(f"  largest revision                                  "
      f"{moved.abs().max():>12,.0f} HK$m")
print(f"  median as a share of that day's balance           "
      f"{100*(moved.abs()/hk['aggregate_balance']).median():>11.2f}%")
print(f"  revised UP / revised DOWN                         "
      f"{(moved > 0).sum():>7} / {(moved < 0).sum()}")

# That last line is the one to think about before you go on. If it is lopsided,
# the settled figure is not a noisy version of the forecast -- it is a biased
# one, and a bias does not average away over a long sample the way noise does.


Now the test that matters. This morning's slide claimed a mechanism: a smaller
Aggregate Balance tightens interbank liquidity and pushes HIBOR up.

Measure it twice — once with the number you would have had, once with the number
you would not. Across four tenors, because the Aggregate Balance is *overnight*
money and the mechanism should not be equally visible at every maturity.

**Do not assume you know which way the damage runs.** Look-ahead has a
reputation for making results better than they were. That is one of the things
it can do.


In [ ]:
hib = (pd.DataFrame(H["hibor"]).T.rename_axis("date")
         .apply(pd.to_numeric, errors="coerce"))
hib.index = pd.to_datetime(hib.index)

SERIES = (("point-in-time", "forecast_for_today_made_yesterday"),   # yesterday's forecast
          ("settled",       "aggregate_balance"))                   # not knowable yet

print("  corr( change in Aggregate Balance , change in HIBOR )\n")
print(f"  {'tenor':<14}{'point-in-time':>16}{'settled':>16}{'gap':>10}")
for TENOR in ("ir_overnight", "ir_1w", "ir_1m", "ir_3m"):
    d = hk.set_index("date").join(hib[[TENOR]], how="inner").sort_index()
    corrs = []
    for _, col in SERIES:
        x, yv = d[col].diff().values, d[TENOR].diff().values
        m = np.isfinite(x) & np.isfinite(yv)
        corrs.append(np.corrcoef(x[m], yv[m])[0, 1])
    print(f"  {TENOR:<14}{corrs[0]:>+16.3f}{corrs[1]:>+16.3f}{corrs[1]-corrs[0]:>+10.3f}")

# The Aggregate Balance is overnight interbank liquidity. Start at ir_overnight
# and read across; the longer tenors are there so you can see the mechanism fade.


In [ ]:
EXTENSION = {
    "pct_days_revised":                  None,  # from the cell above
    "which_direction_were_the_revisions": "",   # and what that rules out
    "does_the_choice_change_the_answer": "",    # compare the two correlations
    "which_way_did_the_error_run":       "",    # did look-ahead flatter the result,
                                                # or damage it? Say why that happened.
    "which_series_is_the_honest_one":    "",    # and why, in one line
    "what_the_FOMC_file_had_that_this_one_does_not": "",
}
_m = [k for k, v in EXTENSION.items() if v is None or (isinstance(v, str) and not v.strip())]
print("Extension complete." if not _m else "Still to fill: " + ", ".join(_m))


**The transferable point, and it is the one to carry into Class 8.** The FOMC
corpus told you there were two dates. It had a field called `release_date`, a
field called `release_lag_days`, and a note saying which to use was your
decision. This file tells you too — but only because somebody wrote the note.
The HKMA API does not.

Most data does not come with a note. It comes with a column called
`aggregate_balance`, and the question of when that number became knowable is one
you have to think to ask.
